# Gemini Function Calling + Unsiloed: Visual Document Audit Trail

Give [Gemini](https://ai.google.dev/gemini-api/docs/function-calling) structured access to [Unsiloed's](https://docs.unsiloed.ai) document processing API, then use Unsiloed's own evidence (per-field confidence scores and word-level bounding-box citations) to draw a visual audit trail on the source document.

Unsiloed's own docs frame the product around one idea: generic OCR and text-only LLM parsers can produce a plausible-looking answer with no way to check it, and that failure mode is exactly what regulated-industry document work (finance, legal, healthcare) cannot tolerate. Confidence scores and bounding boxes are how Unsiloed backs every value with a citation. This notebook makes that citation visible instead of leaving it as an unused JSON field.

**What we'll build:**
1. Gemini function-call schemas for Unsiloed's parse, extract, and classify operations
2. A tool executor that dispatches Gemini's function calls to the Unsiloed API
3. A manual agentic loop: Gemini calls a tool, we execute it, we feed the result back
4. Extraction with citations enabled, showing the dual confidence score (`grounding_score` and `extraction_score`) per field
5. A reusable renderer (`utils/audit_renderer.py`) that draws each field's citation box on the source document, labeled with its raw scores
6. What the scores do when a requested field is not in the document at all
7. The same pipeline against a photographed passport, where the document is hard and one field comes back ungrounded
8. A short classification example

This recipe covers parse, extract, and classify, with extraction citations visualized as an audit trail. It does not cover split, because nothing here needed a multi-document batch, and forcing it in for coverage's sake would have padded the notebook without adding to the story.

## 1. Setup

Install dependencies and load API keys from your `.env` file.

In [ ]:
%pip install google-genai requests python-dotenv pymupdf pillow -q

In [ ]:
import os
import json
import time
import pathlib

import requests
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
UNSILOED_API_KEY = os.getenv("UNSILOED_API_KEY")
UNSILOED_BASE_URL = "https://prod.visionapi.unsiloed.ai"
UNSILOED_HEADERS = {"api-key": UNSILOED_API_KEY}

client = genai.Client(api_key=GEMINI_API_KEY)

# Repo-root-relative sample document, already used elsewhere in the cookbook.
SAMPLE_PDF = pathlib.Path("../../sample-documents/sample-statement.pdf")

print("Setup complete.")

## 2. Define Unsiloed Tools for Gemini

Three tools map to Unsiloed's core operations, plus a poller for the async job pattern every operation shares. Each tool takes a local `file_path` rather than a public URL, so the sample document in this repo works directly.

`extract` deliberately does **not** expose `enable_citations` or `model` as parameters Gemini can set. Per Unsiloed's extract reference, leaving `enable_citations` off returns a zeroed-out `grounding_score` and a `null` citation, and the `alpha` model tier returns a different, legacy response shape. Since this recipe's entire point is the citation evidence, those are fixed to `enable_citations=True, model="gamma"` in the executor below rather than left to chance.

| Tool | Unsiloed endpoint | Purpose |
|------|-------------------|---------|
| `unsiloed_parse_document` | `POST /parse` | Read a document into Markdown chunks |
| `unsiloed_extract_data` | `POST /v2/extract` | Pull typed fields with confidence + citations |
| `unsiloed_classify_document` | `POST /classify` | Predict a document's category |
| `unsiloed_get_job_result` | `GET /{parse,extract,classify}/{job_id}` | Poll an async job to completion |

In [ ]:
tool_declarations = [
    types.FunctionDeclaration(
        name="unsiloed_parse_document",
        description=(
            "Parse a local document into structured Markdown chunks. Use this when the "
            "user wants to read a document or get a summary of its contents. The "
            "operation is asynchronous: it returns a job_id that must be polled with "
            "unsiloed_get_job_result."
        ),
        parameters_json_schema={
            "type": "object",
            "properties": {
                "file_path": {
                    "type": "string",
                    "description": "Local path to the document to parse.",
                }
            },
            "required": ["file_path"],
        },
    ),
    types.FunctionDeclaration(
        name="unsiloed_extract_data",
        description=(
            "Extract specific structured fields from a local PDF using a JSON schema. "
            "Use this when the user names the fields they want (for example: account "
            "holder, account number, closing balance). The operation is asynchronous: it "
            "returns a job_id that must be polled with unsiloed_get_job_result. Do not "
            "use this for classification."
        ),
        parameters_json_schema={
            "type": "object",
            "properties": {
                "file_path": {
                    "type": "string",
                    "description": "Local path to the document to extract from. PDFs, scans and photos are all accepted.",
                },
                "schema_data": {
                    "type": "string",
                    "description": (
                        'A JSON-stringified schema defining the fields to extract. '
                        'Example: {"type":"object","properties":{"total":{"type":"string"}},'
                        '"required":["total"],"additionalProperties":false}'
                    ),
                },
            },
            "required": ["file_path", "schema_data"],
        },
    ),
    types.FunctionDeclaration(
        name="unsiloed_classify_document",
        description=(
            "Classify a local PDF into one of several candidate categories. Use this "
            "when the user wants to know what kind of document this is. The operation "
            "is asynchronous: it returns a job_id that must be polled with "
            "unsiloed_get_job_result. Do not use this for data extraction."
        ),
        parameters_json_schema={
            "type": "object",
            "properties": {
                "file_path": {
                    "type": "string",
                    "description": "Local path to the document to classify.",
                },
                "categories": {
                    "type": "string",
                    "description": (
                        'A JSON-stringified array of category objects, each with a '
                        '"name" and optional "description". Example: '
                        '[{"name":"Invoice"},{"name":"Bank Statement"}]'
                    ),
                },
            },
            "required": ["file_path", "categories"],
        },
    ),
    types.FunctionDeclaration(
        name="unsiloed_get_job_result",
        description=(
            "Poll for the result of an asynchronous Unsiloed job started by "
            "unsiloed_parse_document, unsiloed_extract_data, or "
            "unsiloed_classify_document. If the job is still processing, wait a couple "
            "of seconds and call this again."
        ),
        parameters_json_schema={
            "type": "object",
            "properties": {
                "job_id": {
                    "type": "string",
                    "description": "The job_id returned by a previous Unsiloed tool call.",
                },
                "job_type": {
                    "type": "string",
                    "enum": ["parse", "extract", "classify"],
                    "description": "Which kind of job this is, to poll the right endpoint.",
                },
            },
            "required": ["job_id", "job_type"],
        },
    ),
]

unsiloed_tool = types.Tool(function_declarations=tool_declarations)

print(f"Defined {len(tool_declarations)} tools: {[t.name for t in tool_declarations]}")

## 3. Build the Tool Executor

This function takes a tool name and arguments from Gemini and makes the corresponding HTTP request to the Unsiloed API. `extract` always sends `enable_citations=true` and `model=gamma`, hardcoded rather than left for Gemini to set, so this recipe can't regress into the exact bug a past cookbook PR was caught shipping: citations silently off, confidence scores silently zero.

In [ ]:
def process_tool_call(name: str, args: dict) -> dict:
    """Execute one Unsiloed tool call and return the raw JSON response."""

    if name == "unsiloed_parse_document":
        with open(args["file_path"], "rb") as fh:
            return requests.post(
                f"{UNSILOED_BASE_URL}/parse",
                headers=UNSILOED_HEADERS,
                files={"file": fh},
                timeout=180,
            ).json()

    if name == "unsiloed_extract_data":
        with open(args["file_path"], "rb") as fh:
            return requests.post(
                f"{UNSILOED_BASE_URL}/v2/extract",
                headers=UNSILOED_HEADERS,
                files={"pdf_file": fh},
                data={
                    "schema_data": args["schema_data"],
                    "model": "gamma",
                    "enable_citations": "true",
                },
                timeout=180,
            ).json()

    if name == "unsiloed_classify_document":
        with open(args["file_path"], "rb") as fh:
            return requests.post(
                f"{UNSILOED_BASE_URL}/classify",
                headers=UNSILOED_HEADERS,
                files={"pdf_file": fh},
                data={"categories": args["categories"]},
                timeout=180,
            ).json()

    if name == "unsiloed_get_job_result":
        endpoint = args["job_type"]  # "parse" | "extract" | "classify"
        return requests.get(
            f"{UNSILOED_BASE_URL}/{endpoint}/{args['job_id']}",
            headers=UNSILOED_HEADERS,
            timeout=30,
        ).json()

    raise RuntimeError(f"Unknown tool: {name}")

## 4. Build the Agentic Loop

The same pattern as the sibling `claude/tool-use` notebook, translated to Gemini's manual function-calling API (`automatic_function_calling` disabled, since we want to inspect and print each tool call rather than let the SDK run it silently):

```
User message → Gemini → function_call? → execute tool → feed result back → Gemini → ... → final text
```

One thing worth calling out: Unsiloed's async jobs don't share one status vocabulary. Parse jobs report `Succeeded`/`Failed`; extract and classify jobs report `completed`/`failed`. The system instruction below tells Gemini both, and tells it to treat anything else as "still running" rather than assuming only two values exist, since a past review on this cookbook found a status this notebook does not otherwise handle.

In [ ]:
SYSTEM_INSTRUCTION = (
    "You have tools that call Unsiloed's document API. Every operation is asynchronous: "
    "after submitting a job, call unsiloed_get_job_result to poll it. Parse jobs report "
    "status as 'Succeeded' or 'Failed'. Extract and classify jobs report 'completed' "
    "or 'failed'. Treat any other status as still running: wait a moment and poll "
    "again rather than assuming the job is done. Once you have the final result, "
    "summarize it in plain language for the user instead of repeating raw JSON."
)

GEMINI_CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM_INSTRUCTION,
    tools=[unsiloed_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def run_agent(user_message: str, model: str = "gemini-3.6-flash") -> str:
    """Run an agentic loop where Gemini processes documents via Unsiloed tools."""
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=user_message)])]

    print(f"User: {user_message}\n")

    while True:
        response = client.models.generate_content(model=model, contents=contents, config=GEMINI_CONFIG)
        candidate = response.candidates[0]
        contents.append(candidate.content)

        calls = [p.function_call for p in candidate.content.parts if p.function_call]
        if not calls:
            print(f"Gemini: {response.text}\n")
            return response.text

        response_parts = []
        for call in calls:
            args = call.args or {}
            print(f"  -> {call.name}({args})")
            result = process_tool_call(call.name, args)
            response_parts.append(types.Part.from_function_response(name=call.name, response=result))

        contents.append(types.Content(role="user", parts=response_parts))

## 5. Example: Parse and Summarize

Ask Gemini to read the sample bank statement and summarize it. Gemini will:
1. Call `unsiloed_parse_document`
2. Poll with `unsiloed_get_job_result` until the parse job succeeds
3. Read through the Markdown chunks and summarize them

Requires a real `UNSILOED_API_KEY` and `GEMINI_API_KEY` in your `.env`.

In [ ]:
result = run_agent(
    f"Parse this document and summarize what it says: {SAMPLE_PDF}"
)

## 6. Example: Extract with Citations

Ask Gemini to pull specific fields out of the statement. Under the hood, `extract` always runs with `enable_citations=true` and `model=gamma` (fixed in the executor, not left to Gemini), so every field below comes back with real evidence attached.

In [ ]:
STATEMENT_SCHEMA = {
    "type": "object",
    "properties": {
        "account_holder": {
            "type": "string",
            "description": "Name of the account holder, as written.",
        },
        "account_number": {
            "type": "string",
            "description": "Account number, digits and dashes as written.",
        },
        "closing_balance": {
            "type": "string",
            "description": "Closing balance amount, digits only with a decimal point, no currency symbol.",
        },
        "next_payment_amount": {
            "type": "string",
            "description": "The next scheduled payment amount, digits only with a decimal point.",
        },
        "next_payment_date": {
            "type": "string",
            "description": "Date the next payment is due, ISO 8601 (YYYY-MM-DD).",
        },
    },
    "required": ["account_holder", "account_number", "closing_balance"],
    "additionalProperties": False,
}

result = run_agent(
    f"Extract the account holder, account number, closing balance, and next payment "
    f"amount and date from this statement: {SAMPLE_PDF}. "
    f"Use this schema: {json.dumps(STATEMENT_SCHEMA)}"
)

## 7. The Evidence Behind the Values: Dual Confidence Scores

Gemini's answer above reads naturally, but it is a summary -- the field it is built from carries more than one number. Per Unsiloed's extract reference, each field comes back as `{"value": ..., "score": {"grounding_score": ..., "extraction_score": ...},
"citation": {...}}`: `grounding_score` is confidence the value was actually located in the document, `extraction_score` is confidence it was read correctly once located. Most integrations only show a single flattened number; this recipe surfaces both, plus the citation's bounding box, since that pair is the evidence a reviewer would actually want to check.

This step calls Unsiloed directly rather than through Gemini, so we can inspect the raw result instead of Gemini's paraphrase of it.

In [ ]:
def poll_until_done(job_id: str, job_type: str, max_attempts: int = 40, delay: float = 3.0) -> dict:
    """Poll an Unsiloed job until it reaches a terminal status."""
    terminal = {"Succeeded", "Failed"} if job_type == "parse" else {"completed", "failed"}
    for _ in range(max_attempts):
        body = process_tool_call("unsiloed_get_job_result", {"job_id": job_id, "job_type": job_type})
        if body.get("status") in terminal:
            return body
        time.sleep(delay)
    raise RuntimeError(f"Unsiloed {job_type} job {job_id} did not finish in time")


def run_extract(file_path: str, schema: dict) -> dict:
    """Submit + poll an extraction, returning the raw {value, score, citation} result tree."""
    submit = process_tool_call("unsiloed_extract_data", {
        "file_path": file_path,
        "schema_data": json.dumps(schema),
    })
    job_id = submit.get("job_id")
    if not job_id:
        raise RuntimeError(f"no job_id from Unsiloed extract: {submit}")
    body = poll_until_done(job_id, "extract")
    if body.get("status") == "failed":
        raise RuntimeError(f"Unsiloed extract failed: {body.get('error') or body.get('message')}")
    return body.get("result") or {}


def print_evidence(result: dict) -> None:
    """Print each field's value alongside its raw grounding/extraction scores and citation."""
    for field, leaf in result.items():
        if not isinstance(leaf, dict) or "value" not in leaf:
            continue
        score = leaf.get("score") or {}
        citation = leaf.get("citation")
        loc = f"page {citation['page']}, bbox {citation['bbox']}" if citation else "no citation"
        print(f"{field}: {leaf['value']!r}")
        print(f"  grounding={score.get('grounding_score')}  extraction={score.get('extraction_score')}  ({loc})")

In [ ]:
standard_result = run_extract(str(SAMPLE_PDF), STATEMENT_SCHEMA)
print_evidence(standard_result)

## 8. Visual Audit Trail

The evidence from section 7 is only useful if a reviewer can actually check it against the document. `utils/audit_renderer.py` draws every field's citation box on the source page, labeled with its raw scores -- reusable by any future recipe that needs to show its work, not just this notebook.

In [ ]:
from utils.audit_renderer import render_extraction_audit

audit_image = render_extraction_audit(str(SAMPLE_PDF), standard_result)
audit_image

## 9. The Case the Audit Trail Is Actually For

Everything above scored highly, which makes the evidence easy to ignore. The evidence earns its place on the fields that go wrong.

Below, the schema asks for two fields that are not in this statement at all: an IBAN and a UK sort code. Nothing in the document can satisfy them.

In [ ]:
ABSENT_FIELD_SCHEMA = {
    "type": "object",
    "properties": {
        "account_holder": {"type": "string", "description": "Name of the account holder, as written."},
        "iban": {"type": "string", "description": "International Bank Account Number (IBAN)."},
        "sort_code": {"type": "string", "description": "UK sort code, format 00-00-00."},
    },
    "required": ["account_holder"],
    "additionalProperties": False,
}

absent_result = run_extract(str(SAMPLE_PDF), ABSENT_FIELD_SCHEMA)
print_evidence(absent_result)

### Read the right score

Two things to take from that output.

**Unsiloed returns `null` rather than inventing a plausible IBAN.** Its own [docs](https://www.unsiloed.ai/docs) make fabrication-resistance the pitch; here that holds up under a direct test. Note that both absent fields were left out of `required` in the schema above. A field marked `required` that the document cannot satisfy puts pressure on the model to produce something, which is the failure this whole recipe exists to make visible.

**`extraction_score` stayed high on both absent fields, while `grounding_score` dropped to `0.0`.** That pairing is the useful part. `grounding_score` answers "was this located in the document", and it is the number that catches a missing value; `extraction_score` answers "was it read correctly once located", and on its own it says almost nothing about whether the field is real. Any integration that flattens the two into a single confidence number, or reads only `extraction_score`, can display high confidence for a value that does not exist.

The renderer lists ungrounded fields under the page rather than omitting them, so a field the API declined to ground stays visible instead of quietly disappearing from the audit image.

In [ ]:
render_extraction_audit(str(SAMPLE_PDF), absent_result)

## 10. A Harder Document

The statement is a clean, digitally generated PDF, which is the easy case. This section runs the same pipeline against `kyc-app/samples/passport_specimen.jpg`, already in this repo: a photographed specimen passport, shot at an angle, with holographic glare across the page, trilingual field labels, and a handwritten signature. It is a specimen document, so the details on it are not anyone's.

Three things worth watching in the output.

**The endpoint takes images.** Despite the `pdf_file` form field name, `/v2/extract` accepts photos and scans. Citations for an image come back in pixel coordinates rather than PDF points, and the renderer scales by the ratio of rendered size to the reported `page_width`/`page_height`, so both work without a special case.

**Handwriting.** Reading a signature is where a general-purpose vision model will confidently invent a plausible name.

**A third score signature.** Section 9 showed absent fields returning `grounding_score: 0.0` with `extraction_score` staying high. Watch what the passport number does here, and note that it differs.

In [ ]:
PASSPORT = pathlib.Path("../../kyc-app/samples/passport_specimen.jpg")

PASSPORT_SCHEMA = {
    "type": "object",
    "properties": {
        "surname": {"type": "string", "description": "Surname / Nom, as written."},
        "given_names": {"type": "string", "description": "Given names / Prenoms, as written."},
        "passport_number": {"type": "string", "description": "Passport number."},
        "date_of_birth": {"type": "string", "description": "Date of birth, ISO 8601 (YYYY-MM-DD)."},
        "date_of_expiry": {"type": "string", "description": "Date of expiry, ISO 8601 (YYYY-MM-DD)."},
        "holder_signature": {"type": "string", "description": "The handwritten holder signature text."},
    },
    "additionalProperties": False,
}

passport_result = run_extract(str(PASSPORT), PASSPORT_SCHEMA)
print_evidence(passport_result)

In [ ]:
render_extraction_audit(str(PASSPORT), passport_result)

### What the passport number tells you

The dates come back in ISO form, which means the extractor resolved `01 06 1991` as day-month-year rather than month-day-year. The signature is read from handwriting. Both carry citations you can check against the boxes above.

The passport number is the interesting one: its number is not legible in this crop, and **both** scores come back at `0.0`, where the absent fields in section 9 held a high `extraction_score`. Neither case produced a fabricated value, and in both the `grounding_score` is what tells you not to trust the field. Treat `0.0` as "the API is not standing behind this", and route it to a human rather than into a database.

## 11. Classification

The third operation, kept short. Classification is what you reach for before extraction, when a pile of mixed documents needs routing and you do not yet know which schema each one deserves.

This goes back through `run_agent`, so Gemini picks the tool and polls the job itself rather than us calling the endpoint directly.

In [ ]:
CATEGORIES = [
    {"name": "Bank statement", "description": "Account statements listing balances and transactions"},
    {"name": "Identity document", "description": "Passports, ID cards and driving licences"},
    {"name": "Invoice", "description": "Bills issued for goods or services"},
    {"name": "Contract", "description": "Legal agreements between parties"},
]

result = run_agent(
    f"What kind of document is this? Classify it into one of these categories: "
    f"{json.dumps(CATEGORIES)}. The file is at {PASSPORT}"
)

## Where to go next

The pieces worth reusing outside this notebook:

- **`utils/audit_renderer.py`** is standalone. Hand it any `/v2/extract` result with citations enabled and it renders the evidence, whatever produced the extraction.
- **The tool declarations in section 2** are ordinary JSON Schema, so they port to any model with function calling. Only the `types.FunctionDeclaration` wrapper is Gemini's.
- **`enable_citations=true` and `model=gamma`** are fixed in the executor rather than exposed as parameters. Without citations the grounding pass does not run, and the scores this recipe is built on come back as zeros.

The habit this recipe is really arguing for: read `grounding_score` before you trust a value, and give a human the fields where it is low or zero.

- [Unsiloed API reference](https://www.unsiloed.ai/docs/api-reference/extraction/extract-data)
- [Gemini function calling](https://ai.google.dev/gemini-api/docs/function-calling)
- [Claude tool use with Unsiloed](../../claude/tool-use/) for the same integration against a different model